# Phase 10: Reusable Prediction Pipeline

## 1. Objective
The objective of this phase is to build and demonstrate a reusable, leakage-safe prediction/inference pipeline for Disease Diagnosis Prediction. 
The pipeline uses the persisted final model artifact (`models/final_model.joblib`), which contains the complete preprocessing transformer and tuned Support Vector Machine classifier.

### Key Capabilities & Scope:
1. **Single Prediction**: Accepts raw feature dictionary with all 10 schema features and returns structured result with predicted class, predicted probability for the positive class (class 1), and model metadata.
2. **Batch Prediction**: Processes pandas DataFrames, validates schema, preserves feature ordering, and appends prediction outputs without mutating the original input DataFrame.
3. **Input Validation & Consistency**: Validates data types, required feature presence, boolean format, and handles missing/unrecorded values (`chol == 0` mapped to `NaN`) consistently with the training workflow data cleaning.

> **Disclaimer**: This pipeline provides machine-learning predictions based on the trained model and is not a clinically validated diagnostic system.


## 2. Load Persisted Model
Load the persisted scikit-learn Pipeline artifact from `models/final_model.joblib` and verify its structure.


In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

# Ensure project root is in path
sys.path.append("..")

from src.predict import (
    load_persisted_model,
    load_model_metadata,
    validate_prediction_input,
    prepare_prediction_dataframe,
    predict_single,
    predict_batch,
)
from src.data_loader import load_model_data, split_data

# Load persisted model pipeline and metadata
model_pipeline = load_persisted_model()
metadata = load_model_metadata()

print("Loaded Model Artifact successfully.")
print(f"Model Name: {metadata.get('model_name')}")
print(f"Pipeline Steps: {list(model_pipeline.named_steps.keys())}")
print(f"Classifier: {model_pipeline.named_steps['model']}")


Loaded Model Artifact successfully.
Model Name: Tuned Support Vector Machine
Pipeline Steps: ['preprocessor', 'model']
Classifier: SVC(C=100, kernel='linear', probability=True, random_state=42)


## 3. Define Sample Input
Define a single observation using the exact 10 raw dataset features in standard dictionary format.


In [2]:
sample_input = {
    "age": 55,
    "sex": "Male",
    "cp": "asymptomatic",
    "trestbps": 140,
    "chol": 250,
    "fbs": False,
    "restecg": "normal",
    "thalch": 150,
    "exang": False,
    "oldpeak": 1.2
}

print("Sample Input Observation:")
for k, v in sample_input.items():
    print(f"  {k}: {v}")


Sample Input Observation:
  age: 55
  sex: Male
  cp: asymptomatic
  trestbps: 140
  chol: 250
  fbs: False
  restecg: normal
  thalch: 150
  exang: False
  oldpeak: 1.2


## 4. Validate Input
Validate input schema, presence of all 10 required features, and data types before inference.


In [3]:
df_validated = validate_prediction_input(sample_input)
print("Input validation passed cleanly.")
print("Validated DataFrame shape:", df_validated.shape)
display(df_validated)


Input validation passed cleanly.
Validated DataFrame shape: (1, 10)


,age,sex,cp,trestbps,chol,fbs,restecg,thalch,exang,oldpeak
0,55,Male,asymptomatic,140,250,False,normal,150,False,1.2


## 5. Generate Prediction
Run `predict_single()` to obtain the predicted class label (0: No Disease, 1: Disease Present).


In [4]:
single_result = predict_single(sample_input, model=model_pipeline)

print("Prediction Output Result:")
print(f"  Predicted Class: {single_result['predicted_class']}")
print(f"  Model Name: {single_result['model_name']}")


Prediction Output Result:
  Predicted Class: 1
  Model Name: Tuned Support Vector Machine


## 6. Generate Positive-Class Probability
Extract and display the model's predicted probability for the positive class (class 1).


In [5]:
print(f"Predicted Probability for Positive Class (Class 1): {single_result['predicted_probability']:.4f}")
print("Formatted Output Dictionary:")
display(single_result)


Predicted Probability for Positive Class (Class 1): 0.6461
Formatted Output Dictionary:


{'predicted_class': 1,
 'predicted_probability': 0.6461265923693491,
 'model_name': 'Tuned Support Vector Machine'}

## 7. Demonstrate Batch Prediction
Pass a batch of observations (from held-out test data) to `predict_batch()`.
Verify that the output DataFrame contains `predicted_class` and `predicted_probability`, and that the input DataFrame is NOT mutated.


In [6]:
# Load held-out test dataset observations
X, y = load_model_data()
_, X_test, _, y_test = split_data(X, y)

# Take 10 sample rows
X_batch_input = X_test.head(10).copy()

# Store copy to verify non-mutation
X_batch_copy = X_batch_input.copy()

# Run batch prediction
batch_results = predict_batch(X_batch_input, model=model_pipeline)

print("Batch Prediction Execution Successful.")
print("Output Columns:", list(batch_results.columns))
print("\nSample Batch Output (First 5 Rows):")
display(batch_results[["age", "sex", "cp", "chol", "predicted_class", "predicted_probability"]].head())

# Assert non-mutation of input DataFrame
assert "predicted_class" not in X_batch_input.columns, "Error: Input DataFrame was mutated!"
print("\nNon-mutation check passed: X_batch_input remains unmutated.")


Batch Prediction Execution Successful.
Output Columns: ['age', 'trestbps', 'chol', 'thalch', 'oldpeak', 'sex', 'cp', 'restecg', 'fbs', 'exang', 'predicted_class', 'predicted_probability']

Sample Batch Output (First 5 Rows):


,age,sex,cp,chol,predicted_class,predicted_probability
513,49,Male,asymptomatic,206.0,0,0.369148
824,61,Male,asymptomatic,282.0,1,0.970275
851,66,Male,asymptomatic,261.0,1,0.773243
803,65,Male,asymptomatic,248.0,1,0.958982
886,69,Male,non-anginal,271.0,0,0.411997



Non-mutation check passed: X_batch_input remains unmutated.


## 8. Verify Prediction Reproducibility
Compare direct predictions using `model_pipeline.predict()` and `model_pipeline.predict_proba()` against `src/predict.py` output to confirm 100% exact numerical agreement.


In [7]:
# Direct pipeline inference
df_prep = prepare_prediction_dataframe(X_test)
direct_preds = model_pipeline.predict(df_prep)
direct_probs = model_pipeline.predict_proba(df_prep)[:, 1]

# Pipeline module inference
pipeline_batch = predict_batch(X_test, model=model_pipeline)
pipeline_preds = pipeline_batch["predicted_class"].values
pipeline_probs = pipeline_batch["predicted_probability"].values

# Assert exact numerical equivalence
preds_match = np.array_equal(direct_preds, pipeline_preds)
probs_match = np.allclose(direct_probs, pipeline_probs, atol=1e-6)

print(f"Direct vs Pipeline Class Predictions Match 100%: {preds_match}")
print(f"Direct vs Pipeline Positive-Class Probabilities Match 100%: {probs_match}")

assert preds_match and probs_match, "Reproducibility verification failed!"
print("Reproducibility verification PASSED cleanly.")


Direct vs Pipeline Class Predictions Match 100%: True
Direct vs Pipeline Positive-Class Probabilities Match 100%: True
Reproducibility verification PASSED cleanly.


## 9. Demonstrate Invalid-Input & Chol=0 Handling
Demonstrate how missing features, invalid data types, and `chol=0` (treated as missing/unrecorded) are handled safely.


In [8]:
# 1. Demonstration of chol=0 handling
zero_chol_input = sample_input.copy()
zero_chol_input["chol"] = 0

print("Testing input with chol=0 (unrecorded cholesterol):")
zero_chol_res = predict_single(zero_chol_input, model=model_pipeline)
print(f"  Predicted Class: {zero_chol_res['predicted_class']}")
print(f"  Predicted Probability: {zero_chol_res['predicted_probability']:.4f}")
print("  Explanation: chol=0 is converted to NaN and median-imputed by the pipeline preprocessor.")

# 2. Demonstration of missing feature handling
missing_feature_input = sample_input.copy()
del missing_feature_input["age"]

try:
    predict_single(missing_feature_input)
except ValueError as e:
    print(f"\nCaught expected error for missing feature: {e}")

# 3. Demonstration of invalid numeric input handling
invalid_numeric_input = sample_input.copy()
invalid_numeric_input["trestbps"] = "one-forty"

try:
    predict_single(invalid_numeric_input)
except ValueError as e:
    print(f"\nCaught expected error for invalid numeric input: {e}")


Testing input with chol=0 (unrecorded cholesterol):


  Predicted Class: 1
  Predicted Probability: 0.8694
  Explanation: chol=0 is converted to NaN and median-imputed by the pipeline preprocessor.

Caught expected error for missing feature: Input is missing required feature(s): ['age']

Caught expected error for invalid numeric input: Invalid numeric value 'one-forty' for feature 'trestbps' at index 0. Must be a valid integer or float.


## 10. Limitations & Scope

1. **Specific Schema Dependency**: The prediction pipeline strictly expects the 10 raw features specified (`age`, `sex`, `cp`, `trestbps`, `chol`, `fbs`, `restecg`, `thalch`, `exang`, `oldpeak`). Extra columns like `ca`, `thal`, or `slope` are not present in this dataset schema and must not be supplied.
2. **Missing Feature Sensitivity**: Completely missing feature keys will cause validation failures. Unrecorded zero cholesterol values (`chol=0`) are handled automatically by converting to `NaN` and using training-median imputation inside the pipeline.
3. **Non-Clinical System Disclaimer**: This pipeline provides machine-learning predictions based on the trained model and is not a clinically validated diagnostic system.
